Part 4A only initialize the experiment. It do not compute any metrics. Its job is to load everything required by the later parts (4B–4F).

In [12]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip -q install \
gensim \
pyLDAvis \
openpyxl

In [14]:
# ============================================================
# Imports
# ============================================================

import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from gensim import corpora
from gensim.models import LdaModel
from gensim.models import CoherenceModel

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [15]:
# ============================================================
# Project Directory
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/TopicEvalBench"
)

MODEL_DIR = PROJECT_DIR /"models"

print(PROJECT_DIR)

/content/drive/MyDrive/TopicEvalBench


In [16]:
# ============================================================
# Load Configuration
# ============================================================

config_file = PROJECT_DIR /"config/config.json"

with open(config_file, "r", encoding="utf-8") as f:

    CONFIG = json.load(f)

print("Configuration Loaded.")

Configuration Loaded.


In [17]:
# ============================================================
# Reproducibility
# ============================================================

SEED = CONFIG["random_seed"]

random.seed(SEED)

np.random.seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

print("Random Seed =", SEED)

Random Seed = 42


In [18]:
# ============================================================
# Read Configuration
# ============================================================

N_FOLDS = CONFIG["n_folds"]

MIN_TOPICS = CONFIG["topic_range"]["minimum"]

MAX_TOPICS = CONFIG["topic_range"]["maximum"]

TOP_WORDS = CONFIG["top_words"]

print("Folds        :", N_FOLDS)
print("Topic Range  :", MIN_TOPICS, "-", MAX_TOPICS)
print("Top Words    :", TOP_WORDS)

Folds        : 5
Topic Range  : 2 - 100
Top Words    : 20


In [19]:
# ============================================================
# Load Experiment Manifest
# ============================================================

manifest_file = (
    MODEL_DIR /
    "experiment_manifest.csv"
)

manifest = pd.read_csv(
    manifest_file
)

print()

print("Experiments :", len(manifest))

display(manifest.head())


Experiments : 495


,ExperimentID,Fold,Topics,Status,TrainingDocuments,DictionarySize,RandomSeed,Passes,Iterations,Alpha,...,TrainingTimeSeconds,LogPerplexityTrain,ModelDirectory,ModelFile,MetadataFile,ThetaTrainFile,ThetaTestFile,PhiFile,TopicSizesFile,TopWordsFile
0,F1_K002,1,2,Available,240,891,42,20,400,"[0.5, 0.5]",...,3.3123,-6.298535,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
1,F1_K003,1,3,Available,240,891,42,20,400,"[0.3333333432674408, 0.3333333432674408, 0.333...",...,4.6282,-6.214306,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
2,F1_K004,1,4,Available,240,891,42,20,400,"[0.25, 0.25, 0.25, 0.25]",...,2.7942,-6.193754,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
3,F1_K005,1,5,Available,240,891,42,20,400,"[0.20000000298023224, 0.20000000298023224, 0.2...",...,6.0153,-6.191917,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
4,F1_K006,1,6,Available,240,891,42,20,400,"[0.1666666716337204, 0.1666666716337204, 0.166...",...,2.5762,-6.224693,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...


In [20]:
# ============================================================
# Verify Manifest
# ============================================================

assert len(manifest) == (
    N_FOLDS *
    (MAX_TOPICS - MIN_TOPICS + 1)
)

assert (
    manifest["Status"] == "Available"
).all()

print("Manifest verification passed.")

Manifest verification passed.


In [21]:
# ============================================================
# Print Line
# ============================================================

def print_line():

    print("=" * 70)

In [22]:
# ============================================================
# Load One Experiment
# ============================================================

def load_experiment(row):

    lda = LdaModel.load(
        row["ModelFile"]
    )

    theta = pd.read_csv(
        row["ThetaTrainFile"]
    )

    phi = pd.read_csv(
        row["PhiFile"]
    )

    topic_sizes = pd.read_csv(
        row["TopicSizesFile"]
    )

    top_words = pd.read_csv(
        row["TopWordsFile"]
    )

    with open(
        row["MetadataFile"],
        "r",
        encoding="utf-8"
    ) as f:

        metadata = json.load(f)

    return {

        "lda": lda,

        "theta": theta,

        "phi": phi,

        "topic_sizes": topic_sizes,

        "top_words": top_words,

        "metadata": metadata

    }

In [23]:
# ============================================================
# Test Loading
# ============================================================

experiment = load_experiment(
    manifest.iloc[0]
)

print(experiment.keys())

dict_keys(['lda', 'theta', 'phi', 'topic_sizes', 'top_words', 'metadata'])


In [24]:

# ============================================================
# Load All Experiments for One Fold
# ============================================================

def load_fold(fold):
    """
    Load all experiments for a single fold.

    Parameters
    ----------
    fold : int
        Fold number (1 to N_FOLDS)

    Returns
    -------
    pandas.DataFrame
        Manifest rows corresponding to the specified fold,
        sorted by the number of topics.
    """

    if fold < 1 or fold > N_FOLDS:
        raise ValueError(
            f"Fold must be between 1 and {N_FOLDS}."
        )

    fold_df = (
        manifest[
            manifest["Fold"] == fold
        ]
        .sort_values("Topics")
        .reset_index(drop=True)
    )

    print(f"Fold : {fold}")
    print(f"Experiments : {len(fold_df)}")

    return fold_df

In [25]:

# ============================================================
# Test load_fold()
# ============================================================

fold1 = load_fold(1)

display(fold1.head())

Fold : 1
Experiments : 99


,ExperimentID,Fold,Topics,Status,TrainingDocuments,DictionarySize,RandomSeed,Passes,Iterations,Alpha,...,TrainingTimeSeconds,LogPerplexityTrain,ModelDirectory,ModelFile,MetadataFile,ThetaTrainFile,ThetaTestFile,PhiFile,TopicSizesFile,TopWordsFile
0,F1_K002,1,2,Available,240,891,42,20,400,"[0.5, 0.5]",...,3.3123,-6.298535,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
1,F1_K003,1,3,Available,240,891,42,20,400,"[0.3333333432674408, 0.3333333432674408, 0.333...",...,4.6282,-6.214306,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
2,F1_K004,1,4,Available,240,891,42,20,400,"[0.25, 0.25, 0.25, 0.25]",...,2.7942,-6.193754,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
3,F1_K005,1,5,Available,240,891,42,20,400,"[0.20000000298023224, 0.20000000298023224, 0.2...",...,6.0153,-6.191917,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...
4,F1_K006,1,6,Available,240,891,42,20,400,"[0.1666666716337204, 0.1666666716337204, 0.166...",...,2.5762,-6.224693,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...,/content/drive/MyDrive/TopicEvalBench/models/F...


In [26]:

# ============================================================
# 4B - Compute Perplexity
# ============================================================

perplexity_rows = []

print_line()
print("Computing Perplexity")
print_line()

for fold in range(1, N_FOLDS + 1):

    fold_df = load_fold(fold)

    for _, row in tqdm(
        fold_df.iterrows(),
        total=len(fold_df),
        desc=f"Fold {fold}"
    ):

        # ------------------------------------------
        # Load metadata
        # ------------------------------------------

        with open(
            row["MetadataFile"],
            "r",
            encoding="utf-8"
        ) as f:

            metadata = json.load(f)

        log_perplexity = metadata.get(
            "log_perplexity_train",
            np.nan
        )

        # Convert log perplexity to perplexity
        if pd.notna(log_perplexity):

            perplexity = float(
                np.exp2(-log_perplexity)
            )

        else:

            perplexity = np.nan

        perplexity_rows.append({

            "ExperimentID": row["ExperimentID"],
            "Fold": row["Fold"],
            "Topics": row["Topics"],

            "LogPerplexityTrain": log_perplexity,
            "Perplexity": perplexity

        })

print_line()
print("Perplexity computation completed.")

Computing Perplexity
Fold : 1
Experiments : 99


Fold 1:   0%|          | 0/99 [00:00<?, ?it/s]

Fold : 2
Experiments : 99


Fold 2:   0%|          | 0/99 [00:00<?, ?it/s]

Fold : 3
Experiments : 99


...

Fold : 4
Experiments : 99


Fold 4:   0%|          | 0/99 [00:00<?, ?it/s]

Fold : 5
Experiments : 99


Fold 5:   0%|          | 0/99 [00:00<?, ?it/s]

Perplexity computation completed.


In [27]:

# ============================================================
# Create Perplexity DataFrame
# ============================================================

perplexity_df = pd.DataFrame(
    perplexity_rows
)

perplexity_df = (
    perplexity_df
    .sort_values(
        ["Fold", "Topics"]
    )
    .reset_index(drop=True)
)

display(
    perplexity_df.head()
)

,ExperimentID,Fold,Topics,LogPerplexityTrain,Perplexity
0,F1_K002,1,2,-6.298535,78.713294
1,F1_K003,1,3,-6.214306,74.249327
2,F1_K004,1,4,-6.193754,73.199113
3,F1_K005,1,5,-6.191917,73.105977
4,F1_K006,1,6,-6.224693,74.785806


In [28]:

# ============================================================
# Save Perplexity Results
# ============================================================

perplexity_file = (
    MODEL_DIR /
    "Perplexity.csv"
)

perplexity_df.to_csv(
    perplexity_file,
    index=False
)

print(
    f"Saved:\n{perplexity_file}"
)

Saved:
/content/drive/MyDrive/TopicEvalBench/models/Perplexity.csv


In [29]:

# ============================================================
# Update Manifest
# ============================================================

manifest = manifest.merge(

    perplexity_df[
        [
            "ExperimentID",
            "LogPerplexityTrain",
            "Perplexity"
        ]
    ],

    on="ExperimentID",
    how="left"

)

manifest_stage4 = (
    MODEL_DIR /
    "experiment_manifest_stage4.csv"
)

manifest.to_csv(
    manifest_stage4,
    index=False
)

print(
    f"Updated Manifest:\n{manifest_stage4}"
)

Updated Manifest:
/content/drive/MyDrive/TopicEvalBench/models/experiment_manifest_stage4.csv


In [30]:

# ============================================================
# Perplexity Summary
# ============================================================

print_line()

print("Perplexity Summary")

print_line()

print(
    perplexity_df[
        [
            "Perplexity",
            "LogPerplexityTrain"
        ]
    ].describe()
)

Perplexity Summary
       Perplexity  LogPerplexityTrain
count  495.000000          495.000000
mean    94.947194           -6.553285
std     13.818931            0.215417
min     70.741148           -6.920964
25%     82.893530           -6.745125
50%     96.375779           -6.590599
75%    107.271650           -6.373187
max    121.176353           -6.144478


Part 4C

Instead of computing one coherence measure at a time, we'll compute all four (u_mass, c_v, c_uci, c_npmi) together while the model, dictionary, corpus, and texts are in memory.

In [31]:
# ============================================================
# 4C - Coherence Analysis
# Imports
# ============================================================

import ast
import pickle

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from gensim.models.ldamodel import LdaModel

In [32]:
# ============================================================
# Helper Functions
# ============================================================

def load_dictionary(fold):
    """
    Load Gensim dictionary for a fold.
    """

    dictionary_path = (
        PROJECT_DIR /
        "dictionary" /
        f"Fold_{fold}" /
        "dictionary.dict"
    )

    return Dictionary.load(str(dictionary_path))


def load_train_corpus(fold):
    """
    Load training corpus (BoW).
    """

    corpus_path = (
        MODEL_DIR /
        f"Fold_{fold}" /
        "train_corpus.pkl"
    )

    with open(corpus_path, "rb") as f:

        corpus = pickle.load(f)

    return corpus


def load_train_texts(fold):
    """
    Load tokenized training documents.
    """

    train_file = (
        PROJECT_DIR /
        "folds" /
        f"Fold_{fold}" /
        "train.xlsx"
    )

    train_df = pd.read_excel(train_file)

    texts = (
        train_df["Tokens"]
        .apply(ast.literal_eval)
        .tolist()
    )

    return texts

In [33]:
# ============================================================
# Compute All Coherence Measures
# ============================================================

def compute_all_coherence(
    lda_model,
    dictionary,
    corpus,
    texts
):
    """
    Compute all four standard coherence measures.
    """

    results = {}

    # ------------------------------------------
    # UMass
    # ------------------------------------------

    results["UMass"] = CoherenceModel(
        model=lda_model,
        corpus=corpus,
        dictionary=dictionary,
        coherence="u_mass"
    ).get_coherence()

    # ------------------------------------------
    # C_V
    # ------------------------------------------

    results["CV"] = CoherenceModel(
        model=lda_model,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v"
    ).get_coherence()

    # ------------------------------------------
    # C_UCI
    # ------------------------------------------

    results["UCI"] = CoherenceModel(
        model=lda_model,
        texts=texts,
        dictionary=dictionary,
        coherence="c_uci"
    ).get_coherence()

    # ------------------------------------------
    # C_NPMI
    # ------------------------------------------

    results["NPMI"] = CoherenceModel(
        model=lda_model,
        texts=texts,
        dictionary=dictionary,
        coherence="c_npmi"
    ).get_coherence()

    return results

In [34]:
# ============================================================
# Compute Coherence for All Models
# ============================================================

coherence_rows = []

print_line()
print("Computing Coherence Measures")
print_line()

for fold in range(1, N_FOLDS + 1):

    print(f"\nLoading Fold {fold}")

    dictionary = load_dictionary(fold)
    corpus = load_train_corpus(fold)
    texts = load_train_texts(fold)

    fold_df = load_fold(fold)

    for _, row in tqdm(
        fold_df.iterrows(),
        total=len(fold_df),
        desc=f"Fold {fold}"
    ):

        lda = LdaModel.load(
            row["ModelFile"]
        )

        scores = compute_all_coherence(
            lda,
            dictionary,
            corpus,
            texts
        )

        coherence_rows.append({

            "ExperimentID": row["ExperimentID"],
            "Fold": row["Fold"],
            "Topics": row["Topics"],

            "UMass": scores["UMass"],
            "CV": scores["CV"],
            "UCI": scores["UCI"],
            "NPMI": scores["NPMI"]

        })

        del lda

    gc.collect()

print_line()
print("Coherence computation completed.")

Computing Coherence Measures

Loading Fold 1
Fold : 1
Experiments : 99


Fold 1:   0%|          | 0/99 [00:00<?, ?it/s]


Loading Fold 2
Fold : 2
Experiments : 99


Fold 2:   0%|          | 0/99 [00:00<?, ?it/s]


Loading Fold 3
Fold : 3
Experiments : 99


...


Loading Fold 4
Fold : 4
Experiments : 99


Fold 4:   0%|          | 0/99 [00:00<?, ?it/s]


Loading Fold 5
Fold : 5
Experiments : 99


Fold 5:   0%|          | 0/99 [00:00<?, ?it/s]

Coherence computation completed.


In [35]:
# ============================================================
# Save Coherence Results
# ============================================================

coherence_df = pd.DataFrame(coherence_rows)

coherence_df = (
    coherence_df
    .sort_values(["Fold", "Topics"])
    .reset_index(drop=True)
)

coherence_file = (
    MODEL_DIR /
    "coherence.csv"
)

coherence_df.to_csv(
    coherence_file,
    index=False
)

print(f"Saved:\n{coherence_file}")

display(coherence_df.head())

Saved:
/content/drive/MyDrive/TopicEvalBench/models/coherence.csv


,ExperimentID,Fold,Topics,UMass,CV,UCI,NPMI
0,F1_K002,1,2,-4.680316,0.308302,-4.338849,-0.128971
1,F1_K003,1,3,-3.103219,0.382957,-2.588816,-0.045408
2,F1_K004,1,4,-2.827298,0.442629,-2.979363,-0.055813
3,F1_K005,1,5,-2.914309,0.430353,-2.637528,-0.026696
4,F1_K006,1,6,-2.840108,0.462751,-2.580816,-0.021140


In [36]:
# ============================================================
# Update Manifest
# ============================================================

manifest_stage4 = manifest.merge(

    coherence_df,

    on=[
        "ExperimentID",
        "Fold",
        "Topics"
    ],

    how="left"

)

stage4_file = (
    MODEL_DIR /
    "experiment_manifest_stage4.csv"
)

manifest_stage4.to_csv(
    stage4_file,
    index=False
)

print(f"Updated manifest:\n{stage4_file}")

Updated manifest:
/content/drive/MyDrive/TopicEvalBench/models/experiment_manifest_stage4.csv


Part 4D Structure

Load test corpus,
Compute test log perplexity for every fold,
Compute average test log perplexity,
Compute RPC,
Save results,
Update Stage-4 Manifest

In [37]:
# ============================================================
# Load Test Corpus
# ============================================================

import pickle

def load_test_corpus(fold):
    """
    Load held-out test corpus for a fold.
    """

    corpus_path = (
        MODEL_DIR /
        f"Fold_{fold}" /
        "test_corpus.pkl"
    )

    with open(corpus_path, "rb") as f:
        corpus = pickle.load(f)

    return corpus

In [38]:
# ============================================================
# Compute Test Log Perplexity
# ============================================================

test_perplexity_rows = []

print_line()
print("Computing Test Log Perplexity")
print_line()

for fold in range(1, N_FOLDS + 1):

    test_corpus = load_test_corpus(fold)

    fold_df = load_fold(fold)

    for _, row in tqdm(
        fold_df.iterrows(),
        total=len(fold_df),
        desc=f"Fold {fold}"
    ):

        lda = LdaModel.load(row["ModelFile"])

        log_perplexity_test = lda.log_perplexity(test_corpus)

        perplexity_test = float(
            np.exp2(-log_perplexity_test)
        )

        test_perplexity_rows.append({

            "ExperimentID": row["ExperimentID"],
            "Fold": row["Fold"],
            "Topics": row["Topics"],

            "LogPerplexityTest": log_perplexity_test,
            "PerplexityTest": perplexity_test

        })

        del lda

    gc.collect()

print_line()
print("Completed.")

Computing Test Log Perplexity
Fold : 1
Experiments : 99


Fold 1:   0%|          | 0/99 [00:00<?, ?it/s]

Fold : 2
Experiments : 99


Fold 2:   0%|          | 0/99 [00:00<?, ?it/s]

Fold : 3
Experiments : 99


...

Fold : 4
Experiments : 99


Fold 4:   0%|          | 0/99 [00:00<?, ?it/s]

Fold : 5
Experiments : 99


Fold 5:   0%|          | 0/99 [00:00<?, ?it/s]

Completed.


In [39]:
# ============================================================
# 4D - Compute Fold-wise RPC
# ============================================================

test_perplexity_df = pd.DataFrame(
    test_perplexity_rows
)

test_perplexity_df = (

    test_perplexity_df

    .sort_values(
        ["Fold", "Topics"]
    )

    .reset_index(
        drop=True
    )
)


In [40]:
# ------------------------------------------------------------
# Compute RPC within each fold
# ------------------------------------------------------------

rpc_df = (

    test_perplexity_df

    .sort_values(
        ["Fold", "Topics"]
    )

    .copy()
)

rpc_df["DeltaLogPerplexity"] = (

    rpc_df

    .groupby("Fold")[
        "LogPerplexityTest"
    ]

    .diff()
)

rpc_df["RPC"] = (

    rpc_df[
        "DeltaLogPerplexity"
    ]

    .abs()
)

print("Fold-wise RPC computed.")

display(
    rpc_df.head(20)
)

Fold-wise RPC computed.


,ExperimentID,Fold,Topics,LogPerplexityTest,PerplexityTest,DeltaLogPerplexity,RPC
0,F1_K002,1,2,-6.603739,97.257601,NaN,NaN
1,F1_K003,1,3,-6.728118,106.014498,-0.124379,0.124379
2,F1_K004,1,4,-6.848727,115.258289,-0.120609,0.120609
3,F1_K005,1,5,-6.998321,127.851162,-0.149595,0.149595
4,F1_K006,1,6,-7.142265,141.265426,-0.143943,0.143943
5,F1_K007,1,7,-7.320977,159.894500,-0.178712,0.178712
6,F1_K008,1,8,-7.426441,172.020954,-0.105464,0.105464
7,F1_K009,1,9,-7.542228,186.396121,-0.115788,0.115788
8,F1_K010,1,10,-7.654327,201.456895,-0.112099,0.112099
9,F1_K011,1,11,-7.767099,217.836078,-0.112772,0.112772


In [41]:
# ============================================================
# Save Fold-wise RPC
# ============================================================

rpc_file = (
    MODEL_DIR /
    "RPC.csv"
)

rpc_df.to_csv(

    rpc_file,

    index=False
)

print(
    f"Saved:\n{rpc_file}"
)

Saved:
/content/drive/MyDrive/TopicEvalBench/models/RPC.csv


In [42]:
# ============================================================
# Update Manifest with Fold-wise RPC
# ============================================================

manifest_stage4 = pd.read_csv(

    MODEL_DIR /
    "experiment_manifest_stage4.csv"
)

# Remove old columns if rerunning

old_rpc_columns = [

    "LogPerplexityTest",
    "PerplexityTest",
    "DeltaLogPerplexity",
    "RPC"
]

manifest_stage4 = manifest_stage4.drop(

    columns=[

        column
        for column in old_rpc_columns
        if column in manifest_stage4.columns

    ],

    errors="ignore"
)

manifest_stage4 = manifest_stage4.merge(

    rpc_df[

        [

            "ExperimentID",
            "Fold",
            "Topics",

            "LogPerplexityTest",
            "PerplexityTest",

            "DeltaLogPerplexity",
            "RPC"

        ]

    ],

    on=[

        "ExperimentID",
        "Fold",
        "Topics"

    ],

    how="left"
)

manifest_stage4.to_csv(

    MODEL_DIR /
    "experiment_manifest_stage4.csv",

    index=False
)

print(
    "Fold-wise RPC added to manifest."
)

Fold-wise RPC added to manifest.


Part 4E Structure

Load Coherence & RPC Results;
Compute Mean UMass;
Compute NAC;
Compute NAP;
Save NAC.csv;
Save NAP.csv;
Update experiment_manifest_stage4.csv;

In [43]:
# ============================================================
# 4E-A - Load Results
# ============================================================

coherence_file = MODEL_DIR / "coherence.csv"
rpc_file = MODEL_DIR / "RPC.csv"

coherence_df = pd.read_csv(coherence_file)
rpc_df = pd.read_csv(rpc_file)

print("Coherence rows :", len(coherence_df))
print("RPC rows       :", len(rpc_df))

Coherence rows : 495
RPC rows       : 495


In [44]:
# ============================================================
# 4E-B - Compute Fold-wise NAC
# ============================================================

nac_df = coherence_df.copy()

# ------------------------------------------------------------
# Absolute coherence values
# ------------------------------------------------------------

nac_df["AbsUMass"] = (

    nac_df["UMass"]
    .abs()
)

nac_df["AbsCV"] = (

    nac_df["CV"]
    .abs()
)

nac_df["AbsUCI"] = (

    nac_df["UCI"]
    .abs()
)

nac_df["AbsNPMI"] = (

    nac_df["NPMI"]
    .abs()
)

# ------------------------------------------------------------
# Normalize within each fold
# ------------------------------------------------------------

nac_df["NAC_UMass"] = (

    nac_df["AbsUMass"]

    /

    nac_df.groupby("Fold")[
        "AbsUMass"
    ].transform("max")
)

nac_df["NAC_CV"] = (

    nac_df["AbsCV"]

    /

    nac_df.groupby("Fold")[
        "AbsCV"
    ].transform("max")
)

nac_df["NAC_UCI"] = (

    nac_df["AbsUCI"]

    /

    nac_df.groupby("Fold")[
        "AbsUCI"
    ].transform("max")
)

nac_df["NAC_NPMI"] = (

    nac_df["AbsNPMI"]

    /

    nac_df.groupby("Fold")[
        "AbsNPMI"
    ].transform("max")
)

display(
    nac_df.head()
)

,ExperimentID,Fold,Topics,UMass,CV,UCI,NPMI,AbsUMass,AbsCV,AbsUCI,AbsNPMI,NAC_UMass,NAC_CV,NAC_UCI,NAC_NPMI
0,F1_K002,1,2,-4.680316,0.308302,-4.338849,-0.128971,4.680316,0.308302,4.338849,0.128971,0.911404,0.574189,0.720333,0.801366
1,F1_K003,1,3,-3.103219,0.382957,-2.588816,-0.045408,3.103219,0.382957,2.588816,0.045408,0.604294,0.713229,0.429793,0.282142
2,F1_K004,1,4,-2.827298,0.442629,-2.979363,-0.055813,2.827298,0.442629,2.979363,0.055813,0.550564,0.824363,0.494632,0.346796
3,F1_K005,1,5,-2.914309,0.430353,-2.637528,-0.026696,2.914309,0.430353,2.637528,0.026696,0.567507,0.801500,0.437881,0.165874
4,F1_K006,1,6,-2.840108,0.462751,-2.580816,-0.021140,2.840108,0.462751,2.580816,0.021140,0.553058,0.861839,0.428465,0.131356


In [45]:
# ============================================================
# Compute Fold-wise NAP
# ============================================================

nap_df = (

    test_perplexity_df[

        [

            "ExperimentID",
            "Fold",
            "Topics",
            "LogPerplexityTest",
            "PerplexityTest"

        ]

    ]

    .copy()
)

nap_df.rename(

    columns={

        "LogPerplexityTest":
        "MeanLogPerplexity"

    },

    inplace=True
)

nap_df["AbsLogPerplexity"] = (

    nap_df[
        "MeanLogPerplexity"
    ]

    .abs()
)

# ------------------------------------------------------------
# Normalize within each fold
# ------------------------------------------------------------

nap_df["NAP"] = (

    nap_df[
        "AbsLogPerplexity"
    ]

    /

    nap_df.groupby("Fold")[
        "AbsLogPerplexity"
    ].transform("max")
)

display(
    nap_df.head()
)

,ExperimentID,Fold,Topics,MeanLogPerplexity,PerplexityTest,AbsLogPerplexity,NAP
0,F1_K002,1,2,-6.603739,97.257601,6.603739,0.480608
1,F1_K003,1,3,-6.728118,106.014498,6.728118,0.489660
2,F1_K004,1,4,-6.848727,115.258289,6.848727,0.498438
3,F1_K005,1,5,-6.998321,127.851162,6.998321,0.509325
4,F1_K006,1,6,-7.142265,141.265426,7.142265,0.519801


In [46]:
# ============================================================
# Save Fold-wise NAC
# ============================================================

nac_file = (
    MODEL_DIR /
    "NAC.csv"
)

nac_df.to_csv(

    nac_file,

    index=False
)

print(
    f"Saved:\n{nac_file}"
)


# ============================================================
# Save Fold-wise NAP
# ============================================================

nap_file = (
    MODEL_DIR /
    "NAP.csv"
)

nap_df.to_csv(

    nap_file,

    index=False
)

print(
    f"Saved:\n{nap_file}"
)

Saved:
/content/drive/MyDrive/TopicEvalBench/models/NAC.csv
Saved:
/content/drive/MyDrive/TopicEvalBench/models/NAP.csv


In [47]:
# ============================================================
# Update Stage-4 Manifest with Fold-wise NAC and NAP
# ============================================================

manifest_stage4 = pd.read_csv(

    MODEL_DIR /
    "experiment_manifest_stage4.csv"
)

# ------------------------------------------------------------
# Remove previous NAC/NAP columns
# ------------------------------------------------------------

old_metric_columns = [

    "AbsUMass",
    "AbsCV",
    "AbsUCI",
    "AbsNPMI",

    "NAC_UMass",
    "NAC_CV",
    "NAC_UCI",
    "NAC_NPMI",

    "MeanLogPerplexity",
    "AbsLogPerplexity",
    "NAP"
]

manifest_stage4 = manifest_stage4.drop(

    columns=[

        column
        for column in old_metric_columns
        if column in manifest_stage4.columns

    ],

    errors="ignore"
)

# ------------------------------------------------------------
# Merge NAC
# ------------------------------------------------------------

manifest_stage4 = manifest_stage4.merge(

    nac_df[

        [

            "ExperimentID",
            "Fold",
            "Topics",

            "AbsUMass",
            "AbsCV",
            "AbsUCI",
            "AbsNPMI",

            "NAC_UMass",
            "NAC_CV",
            "NAC_UCI",
            "NAC_NPMI"

        ]

    ],

    on=[

        "ExperimentID",
        "Fold",
        "Topics"

    ],

    how="left"
)

# ------------------------------------------------------------
# Merge NAP
# ------------------------------------------------------------

manifest_stage4 = manifest_stage4.merge(

    nap_df[

        [

            "ExperimentID",
            "Fold",
            "Topics",

            "MeanLogPerplexity",
            "AbsLogPerplexity",
            "NAP"

        ]

    ],

    on=[

        "ExperimentID",
        "Fold",
        "Topics"

    ],

    how="left"
)

manifest_stage4.to_csv(

    MODEL_DIR /
    "experiment_manifest_stage4.csv",

    index=False
)

print(
    "✓ Stage-4 manifest updated with fold-wise NAC and NAP."
)

display(
    manifest_stage4.head()
)

✓ Stage-4 manifest updated with fold-wise NAC and NAP.


,ExperimentID,Fold,Topics,Status,TrainingDocuments,DictionarySize,RandomSeed,Passes,Iterations,Alpha,...,AbsCV,AbsUCI,AbsNPMI,NAC_UMass,NAC_CV,NAC_UCI,NAC_NPMI,MeanLogPerplexity,AbsLogPerplexity,NAP
0,F1_K002,1,2,Available,240,891,42,20,400,"[0.5, 0.5]",...,0.308302,4.338849,0.128971,0.911404,0.574189,0.720333,0.801366,-6.603739,6.603739,0.480608
1,F1_K003,1,3,Available,240,891,42,20,400,"[0.3333333432674408, 0.3333333432674408, 0.333...",...,0.382957,2.588816,0.045408,0.604294,0.713229,0.429793,0.282142,-6.728118,6.728118,0.489660
2,F1_K004,1,4,Available,240,891,42,20,400,"[0.25, 0.25, 0.25, 0.25]",...,0.442629,2.979363,0.055813,0.550564,0.824363,0.494632,0.346796,-6.848727,6.848727,0.498438
3,F1_K005,1,5,Available,240,891,42,20,400,"[0.20000000298023224, 0.20000000298023224, 0.2...",...,0.430353,2.637528,0.026696,0.567507,0.801500,0.437881,0.165874,-6.998321,6.998321,0.509325
4,F1_K006,1,6,Available,240,891,42,20,400,"[0.1666666716337204, 0.1666666716337204, 0.166...",...,0.462751,2.580816,0.021140,0.553058,0.861839,0.428465,0.131356,-7.142265,7.142265,0.519801
